# Tarea 3 — PyTorch

**Nombre del integrante:** Daniela Zambrano

**Cédula del integrante:** 30956881

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [16]:
# ==========================================
# IMPORTS
# ==========================================

import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from pathlib import Path


# ==========================================
# GENERAL CONFIGURATION
# ==========================================

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==========================================
# DATASET CONFIGURATION
# ==========================================

IMAGE_SIZE = 28
INPUT_NEURONS = IMAGE_SIZE * IMAGE_SIZE

ORIGINAL_CLASSES = 10
SIMPLIFIED_CLASSES = 4

DATA_DIR = Path("./data")


# Original Fashion MNIST class names
ORIGINAL_CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]


# Simplified class names
SIMPLIFIED_CLASS_NAMES = [
    "Top",
    "Bottom",
    "Footwear",
    "Bag"
]


# Mapping from original labels to simplified labels
SIMPLIFIED_LABEL_MAP = {
    0: 0,  # T-shirt/top -> Top
    1: 1,  # Trouser -> Bottom
    2: 0,  # Pullover -> Top
    3: 1,  # Dress -> Bottom
    4: 0,  # Coat -> Top
    5: 2,  # Sandal -> Footwear
    6: 0,  # Shirt -> Top
    7: 2,  # Sneaker -> Footwear
    8: 3,  # Bag -> Bag
    9: 2   # Ankle boot -> Footwear
}


# ==========================================
# NETWORK DEFAULT PARAMETERS
# ==========================================

DEFAULT_HIDDEN_LAYERS = 2
DEFAULT_HIDDEN_NEURONS = 512

DEFAULT_BATCH_SIZE = 128
DEFAULT_LEARNING_RATE = 0.001
DEFAULT_OPTIMIZER = "Adam"

DEFAULT_USE_BATCH_NORM = False
DEFAULT_USE_DROPOUT = False
DEFAULT_DROPOUT_RATE = 0.2
DEFAULT_WEIGHT_DECAY = 0.0
DEFAULT_EARLY_STOPPING = False

NUM_WORKERS = 0


# ==========================================
# MODEL FILES
# ==========================================

MODEL_FILES = {
    "original_rectangular":
        Path("pytorch_fashion_2_original_rectangular.pth"),

    "original_convolutional":
        Path("pytorch_fashion_2_original_convolutional.pth"),

    "simplified_rectangular":
        Path("pytorch_fashion_2_simplified_rectangular.pth"),

    "simplified_convolutional":
        Path("pytorch_fashion_2_simplified_convolutional.pth")
}


# ==========================================
# INITIAL CONFIGURATION
# ==========================================

print("=" * 45)
print("FASHION MNIST - PYTORCH")
print("=" * 45)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Input neurons: {INPUT_NEURONS}")
print(f"Original classes: {ORIGINAL_CLASSES}")
print(f"Simplified classes: {SIMPLIFIED_CLASSES}")

print("=" * 45)

FASHION MNIST - PYTORCH
PyTorch version: 2.15.0+xpu
Device: cpu
Input neurons: 784
Original classes: 10
Simplified classes: 4


## 2. Datos (Fashion MNIST original y clases simplificadas)

In [17]:
# ==========================================
# DATA TRANSFORMATIONS
# ==========================================

transform = transforms.ToTensor()


# ==========================================
# LOAD ORIGINAL FASHION MNIST DATASET
# ==========================================

train_original_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=transform
)

test_original_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transform
)


# ==========================================
# SIMPLIFIED FASHION MNIST DATASET
# ==========================================

class SimplifiedFashionMNIST(torch.utils.data.Dataset):
    """
    Dataset wrapper that groups Fashion MNIST
    labels into four simplified classes.
    """

    def __init__(self, original_dataset, label_map):
        self.original_dataset = original_dataset
        self.label_map = label_map

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, index):
        image, original_label = self.original_dataset[index]

        simplified_label = self.label_map[int(original_label)]

        return image, simplified_label


# Create simplified datasets
train_simplified_dataset = SimplifiedFashionMNIST(
    train_original_dataset,
    SIMPLIFIED_LABEL_MAP
)

test_simplified_dataset = SimplifiedFashionMNIST(
    test_original_dataset,
    SIMPLIFIED_LABEL_MAP
)


# ==========================================
# CREATE DATA LOADERS
# ==========================================

train_original_loader = DataLoader(
    train_original_dataset,
    batch_size=DEFAULT_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_original_loader = DataLoader(
    test_original_dataset,
    batch_size=DEFAULT_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

train_simplified_loader = DataLoader(
    train_simplified_dataset,
    batch_size=DEFAULT_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_simplified_loader = DataLoader(
    test_simplified_dataset,
    batch_size=DEFAULT_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)


# ==========================================
# DATASET INFORMATION
# ==========================================

print("\n" + "=" * 50)
print("FASHION MNIST DATASET INFORMATION")
print("=" * 50)

print(f"Training images: {len(train_original_dataset)}")
print(f"Testing images: {len(test_original_dataset)}")

print("\nOriginal classes:")
for index, name in enumerate(ORIGINAL_CLASS_NAMES):
    print(f"{index}: {name}")

print("\nSimplified classes:")
for index, name in enumerate(SIMPLIFIED_CLASS_NAMES):
    print(f"{index}: {name}")

print("\nSimplified label mapping:")
for original_label, simplified_label in SIMPLIFIED_LABEL_MAP.items():
    print(
        f"{ORIGINAL_CLASS_NAMES[original_label]} "
        f"-> {SIMPLIFIED_CLASS_NAMES[simplified_label]}"
    )

print("\n" + "=" * 50)
print("DATA LOADERS CREATED SUCCESSFULLY")
print("=" * 50)

print(f"Training batches (original): {len(train_original_loader)}")
print(f"Testing batches (original): {len(test_original_loader)}")
print(f"Training batches (simplified): {len(train_simplified_loader)}")
print(f"Testing batches (simplified): {len(test_simplified_loader)}")


# ==========================================
# VERIFY DATA FORMAT
# ==========================================

sample_images, sample_labels = next(iter(train_original_loader))

print("\nData verification:")
print(f"Image batch shape: {sample_images.shape}")
print(f"Label batch shape: {sample_labels.shape}")
print(f"Image pixel range: [{sample_images.min():.1f}, {sample_images.max():.1f}]")

print("\n" + "=" * 50)
print("DATA PREPARATION COMPLETED")
print("=" * 50)



FASHION MNIST DATASET INFORMATION
Training images: 60000
Testing images: 10000

Original classes:
0: T-shirt/top
1: Trouser
2: Pullover
3: Dress
4: Coat
5: Sandal
6: Shirt
7: Sneaker
8: Bag
9: Ankle boot

Simplified classes:
0: Top
1: Bottom
2: Footwear
3: Bag

Simplified label mapping:
T-shirt/top -> Top
Trouser -> Bottom
Pullover -> Top
Dress -> Bottom
Coat -> Top
Sandal -> Footwear
Shirt -> Top
Sneaker -> Footwear
Bag -> Bag
Ankle boot -> Footwear

DATA LOADERS CREATED SUCCESSFULLY
Training batches (original): 469
Testing batches (original): 79
Training batches (simplified): 469
Testing batches (simplified): 79

Data verification:
Image batch shape: torch.Size([128, 1, 28, 28])
Label batch shape: torch.Size([128])
Image pixel range: [0.0, 1.0]

DATA PREPARATION COMPLETED


## 3. Modelos (rectangular y convolucional)

In [18]:
# ==========================================
# HIDDEN LAYER SIZE CALCULATION
# ==========================================

def calculate_hidden_sizes(
    input_size: int,
    output_size: int,
    hidden_layers: int
) -> list:
    """
    Calculates the number of neurons in each hidden layer
    using linear interpolation.
    """

    if hidden_layers < 0:
        raise ValueError("The number of hidden layers cannot be negative.")

    if input_size <= 0 or output_size <= 0:
        raise ValueError("Input and output sizes must be positive.")

    if hidden_layers == 0:
        return []

    hidden_sizes = []

    for i in range(1, hidden_layers + 1):
        neurons = round(
            input_size
            + (output_size - input_size) * i / (hidden_layers + 1)
        )

        if neurons <= 0:
            raise ValueError("Calculated hidden layer size must be positive.")

        hidden_sizes.append(neurons)

    return hidden_sizes


# ==========================================
# NEURAL NETWORK MODEL CREATION
# ==========================================

def build_model(
    architecture: str = "rectangular",
    hidden_layers: int = DEFAULT_HIDDEN_LAYERS,
    hidden_neurons: int = DEFAULT_HIDDEN_NEURONS,
    num_classes: int = ORIGINAL_CLASSES,
    use_batch_norm: bool = DEFAULT_USE_BATCH_NORM,
    use_dropout: bool = DEFAULT_USE_DROPOUT,
    dropout_rate: float = DEFAULT_DROPOUT_RATE
):
    """
    Builds a neural network using nn.Sequential.

    Supported architectures:
    - rectangular: all hidden layers have the same number of neurons.
    - convolutional: hidden layer sizes are calculated using interpolation.

    Returns:
        model: PyTorch Sequential neural network.
        hidden_sizes: List containing the neurons in each hidden layer.
    """

    # Validate parameters
    if architecture not in ("rectangular", "convolutional"):
        raise ValueError("Architecture must be rectangular or convolutional.")

    if hidden_layers < 0:
        raise ValueError("The number of hidden layers cannot be negative.")

    if num_classes not in (ORIGINAL_CLASSES, SIMPLIFIED_CLASSES):
        raise ValueError("The number of output classes must be 10 or 4.")

    if hidden_neurons <= 0:
        raise ValueError("The number of hidden neurons must be positive.")

    if not 0 <= dropout_rate < 1:
        raise ValueError("Dropout rate must be between 0 and 1.")

    # Determine hidden layer sizes
    if architecture == "rectangular":
        hidden_sizes = [hidden_neurons] * hidden_layers

    else:
        hidden_sizes = calculate_hidden_sizes(
            input_size=INPUT_NEURONS,
            output_size=num_classes,
            hidden_layers=hidden_layers
        )

    # Build the Sequential model
    layers = [nn.Flatten()]

    previous_size = INPUT_NEURONS

    for neurons in hidden_sizes:

        # Fully connected layer
        layers.append(nn.Linear(previous_size, neurons))

        # Optional batch normalization
        if use_batch_norm:
            layers.append(nn.BatchNorm1d(neurons))

        # Activation function
        layers.append(nn.ReLU())

        # Optional dropout
        if use_dropout:
            layers.append(nn.Dropout(p=dropout_rate))

        previous_size = neurons

    # Output layer
    layers.append(nn.Linear(previous_size, num_classes))

    # Create and move the model to the selected device
    model = nn.Sequential(*layers).to(DEVICE)

    return model, hidden_sizes


# ==========================================
# MODEL CONFIGURATION VERIFICATION
# ==========================================

print("=" * 55)
print("NEURAL NETWORK ARCHITECTURE VERIFICATION")
print("=" * 55)


# Original rectangular model
rectangular_model, rectangular_sizes = build_model(
    architecture="rectangular",
    hidden_layers=2,
    hidden_neurons=512,
    num_classes=ORIGINAL_CLASSES
)

print("\nOriginal rectangular architecture:")
print(f"784 -> {' -> '.join(map(str, rectangular_sizes))} -> 10")
print(rectangular_model)


# Original convolutional model
convolutional_model, convolutional_sizes = build_model(
    architecture="convolutional",
    hidden_layers=2,
    num_classes=ORIGINAL_CLASSES
)

print("\nOriginal convolutional architecture:")
print(f"784 -> {' -> '.join(map(str, convolutional_sizes))} -> 10")
print(convolutional_model)


# Simplified rectangular model
simplified_rectangular_model, simplified_rectangular_sizes = build_model(
    architecture="rectangular",
    hidden_layers=2,
    hidden_neurons=512,
    num_classes=SIMPLIFIED_CLASSES
)

print("\nSimplified rectangular architecture:")
print(f"784 -> {' -> '.join(map(str, simplified_rectangular_sizes))} -> 4")


# Simplified convolutional model
simplified_convolutional_model, simplified_convolutional_sizes = build_model(
    architecture="convolutional",
    hidden_layers=2,
    num_classes=SIMPLIFIED_CLASSES
)

print("\nSimplified convolutional architecture:")
print(f"784 -> {' -> '.join(map(str, simplified_convolutional_sizes))} -> 4")


print("\n" + "=" * 55)
print("MODEL DEFINITIONS COMPLETED")
print("=" * 55)



NEURAL NETWORK ARCHITECTURE VERIFICATION

Original rectangular architecture:
784 -> 512 -> 512 -> 10
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=512, bias=True)
  (2): ReLU()
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): ReLU()
  (5): Linear(in_features=512, out_features=10, bias=True)
)

Original convolutional architecture:
784 -> 526 -> 268 -> 10
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=526, bias=True)
  (2): ReLU()
  (3): Linear(in_features=526, out_features=268, bias=True)
  (4): ReLU()
  (5): Linear(in_features=268, out_features=10, bias=True)
)

Simplified rectangular architecture:
784 -> 512 -> 512 -> 4

Simplified convolutional architecture:
784 -> 524 -> 264 -> 4

MODEL DEFINITIONS COMPLETED


## 4. Entrenar, probar, guardar y cargar

In [19]:
# ==========================================
# MODEL TRAINING
# ==========================================

def train_model(
    model,
    train_loader,
    epochs: int,
    learning_rate: float = DEFAULT_LEARNING_RATE,
    optimizer_name: str = DEFAULT_OPTIMIZER,
    weight_decay: float = DEFAULT_WEIGHT_DECAY
):
    """
    Trains a neural network using mini-batch gradient descent.

    Returns:
        history: Average training loss for each epoch.
    """

    if epochs <= 0:
        raise ValueError("The number of epochs must be positive.")

    if learning_rate <= 0:
        raise ValueError("Learning rate must be positive.")

    if weight_decay < 0:
        raise ValueError("Weight decay cannot be negative.")

    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    # Select optimizer
    if optimizer_name.lower() == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

    elif optimizer_name.lower() == "sgd":
        optimizer = optim.SGD(
            model.parameters(),
            lr=learning_rate,
            momentum=0.9,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("Supported optimizers are Adam and SGD.")

    history = []

    print("\n" + "=" * 50)
    print("STARTING MODEL TRAINING")
    print("=" * 50)

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # Clear previous gradients
            optimizer.zero_grad()

            # Forward propagation
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)

        # Calculate average loss for the epoch
        epoch_loss = running_loss / total_samples

        history.append(epoch_loss)

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"- Loss: {epoch_loss:.6f}"
        )

    print("\nTRAINING COMPLETED")
    print("=" * 50)

    return history


# ==========================================
# TRAINING LOSS VISUALIZATION
# ==========================================

def plot_loss_history(loss_history):
    """
    Displays the training loss for each epoch.
    """

    if not loss_history:
        raise ValueError("There is no training history to display.")

    plt.figure(figsize=(9, 5))

    plt.plot(
        range(1, len(loss_history) + 1),
        loss_history,
        marker="o",
        linewidth=2
    )

    plt.title("Training Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.xticks(range(1, len(loss_history) + 1))
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# ==========================================
# MODEL EVALUATION
# ==========================================

def evaluate_model(
    model,
    test_loader,
    class_names: list
):
    """
    Evaluates the model and calculates:
    - Confusion matrix
    - Accuracy
    - Macro precision
    - Macro recall

    Returns:
        Dictionary containing evaluation metrics.
    """

    num_classes = len(class_names)

    if num_classes == 0:
        raise ValueError("At least one class name is required.")

    model = model.to(DEVICE)
    model.eval()

    confusion_matrix = np.zeros(
        (num_classes, num_classes),
        dtype=np.int64
    )

    total_samples = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            predictions = torch.argmax(outputs, dim=1)

            # Count actual and predicted class combinations
            batch_counts = torch.bincount(
                labels * num_classes + predictions,
                minlength=num_classes * num_classes
            )

            confusion_matrix += batch_counts.reshape(
                num_classes,
                num_classes
            ).cpu().numpy()

            total_samples += labels.size(0)

    if total_samples == 0:
        raise ValueError("The testing dataset is empty.")

    # Calculate accuracy
    correct_predictions = np.trace(confusion_matrix)

    accuracy = correct_predictions / total_samples

    # Calculate precision for each class
    true_positives = np.diag(confusion_matrix)

    predicted_per_class = confusion_matrix.sum(axis=0)
    actual_per_class = confusion_matrix.sum(axis=1)

    precision_per_class = np.divide(
        true_positives,
        predicted_per_class,
        out=np.zeros(num_classes, dtype=float),
        where=predicted_per_class != 0
    )

    recall_per_class = np.divide(
        true_positives,
        actual_per_class,
        out=np.zeros(num_classes, dtype=float),
        where=actual_per_class != 0
    )

    # Macro averages
    precision = np.mean(precision_per_class)
    recall = np.mean(recall_per_class)

    # Print metrics
    print("\n" + "=" * 55)
    print("MODEL EVALUATION RESULTS")
    print("=" * 55)

    print(f"Total testing images: {total_samples}")
    print(f"Correct predictions: {correct_predictions}")

    print(f"\nAccuracy:  {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")

    print("\nPer-class metrics:")

    for i, class_name in enumerate(class_names):

        print(
            f"{class_name:15s} "
            f"Precision: {precision_per_class[i] * 100:6.2f}% "
            f"Recall: {recall_per_class[i] * 100:6.2f}%"
        )

    print("=" * 55)

    # Plot confusion matrix
    plt.figure(figsize=(9, 7))

    plt.imshow(confusion_matrix, interpolation="nearest", cmap="Blues")

    plt.title("Confusion Matrix")
    plt.colorbar()

    positions = np.arange(num_classes)

    plt.xticks(
        positions,
        class_names,
        rotation=45,
        ha="right"
    )

    plt.yticks(positions, class_names)

    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")

    # Display values inside the matrix
    threshold = confusion_matrix.max() / 2

    for i in range(num_classes):
        for j in range(num_classes):

            value = confusion_matrix[i, j]

            plt.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                color="white" if value > threshold else "black"
            )

    plt.tight_layout()
    plt.show()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "confusion_matrix": confusion_matrix
    }


# ==========================================
# HYPERPARAMETER CONFIGURATION
# ==========================================

def create_hyperparameters(
    architecture: str,
    hidden_layers: int,
    hidden_neurons: int,
    hidden_sizes: list,
    classification: str,
    epochs_trained: int = 0,
    use_batch_norm: bool = False,
    use_dropout: bool = False,
    dropout_rate: float = DEFAULT_DROPOUT_RATE,
    optimizer_name: str = DEFAULT_OPTIMIZER,
    learning_rate: float = DEFAULT_LEARNING_RATE,
    weight_decay: float = DEFAULT_WEIGHT_DECAY
):
    """
    Creates the hyperparameter dictionary required
    when saving a model.
    """

    return {
        "architecture": architecture,
        "network_type": architecture,
        "hidden_layers": hidden_layers,
        "hidden_neurons": hidden_neurons,
        "hidden_sizes": list(hidden_sizes),
        "epochs_trained": epochs_trained,
        "classification": classification,
        "num_classes": (
            ORIGINAL_CLASSES
            if classification == "original"
            else SIMPLIFIED_CLASSES
        ),
        "use_batch_norm": use_batch_norm,
        "use_dropout": use_dropout,
        "dropout_rate": dropout_rate,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "optimizations": {
            "batch_norm": use_batch_norm,
            "dropout": use_dropout,
            "weight_decay": weight_decay > 0
        }
    }


# ==========================================
# MODEL SAVING
# ==========================================

def save_model(model, hyperparameters: dict, path):
    """
    Saves a model using the required .pth format.
    """

    if not isinstance(model, nn.Sequential):
        raise TypeError("The model must be an nn.Sequential instance.")

    required_keys = [
        "architecture",
        "hidden_layers",
        "hidden_sizes",
        "epochs_trained",
        "classification"
    ]

    for key in required_keys:
        if key not in hyperparameters:
            raise ValueError(
                f"Missing required hyperparameter: {key}"
            )

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Store the state dictionary on CPU
    state_dict = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }

    checkpoint = {
        "state_dict": state_dict,
        "hyperparameters": dict(hyperparameters)
    }

    torch.save(checkpoint, path)

    print(f"\nModel saved successfully: {path}")


# ==========================================
# MODEL LOADING
# ==========================================

def load_model(path, device=DEVICE):
    """
    Loads a saved model and reconstructs its architecture
    using the stored hyperparameters.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Model file not found: {path}")

    try:
        checkpoint = torch.load(
            path,
            map_location=device,
            weights_only=True
        )
    except TypeError:
        # Compatibility with older PyTorch versions
        checkpoint = torch.load(
            path,
            map_location=device
        )

    if not isinstance(checkpoint, dict):
        raise ValueError("Invalid model file.")

    if "state_dict" not in checkpoint:
        raise ValueError("The file does not contain a state_dict.")

    if "hyperparameters" not in checkpoint:
        raise ValueError("The file does not contain hyperparameters.")

    hyperparameters = checkpoint["hyperparameters"]

    # Recover the classification type
    classification = hyperparameters["classification"]

    if classification == "original":
        num_classes = ORIGINAL_CLASSES

    elif classification == "simplified":
        num_classes = SIMPLIFIED_CLASSES

    else:
        raise ValueError("Unknown classification type.")

    # Recover the network architecture
    architecture = hyperparameters.get(
        "architecture",
        hyperparameters.get("network_type")
    )

    if architecture not in ("rectangular", "convolutional"):
        raise ValueError("Unknown network architecture.")

    hidden_layers = int(hyperparameters["hidden_layers"])

    hidden_neurons = int(
        hyperparameters.get(
            "hidden_neurons",
            DEFAULT_HIDDEN_NEURONS
        )
    )

    use_batch_norm = bool(
        hyperparameters.get("use_batch_norm", False)
    )

    use_dropout = bool(
        hyperparameters.get("use_dropout", False)
    )

    dropout_rate = float(
        hyperparameters.get(
            "dropout_rate",
            DEFAULT_DROPOUT_RATE
        )
    )

    # Reconstruct the network
    model, hidden_sizes = build_model(
        architecture=architecture,
        hidden_layers=hidden_layers,
        hidden_neurons=hidden_neurons,
        num_classes=num_classes,
        use_batch_norm=use_batch_norm,
        use_dropout=use_dropout,
        dropout_rate=dropout_rate
    )

    # Verify stored hidden layer dimensions
    saved_sizes = hyperparameters.get("hidden_sizes")

    if saved_sizes is not None:
        if list(saved_sizes) != hidden_sizes:
            raise ValueError(
                "Stored hidden layer sizes do not match the architecture."
            )

    # Restore trained parameters
    model.load_state_dict(
        checkpoint["state_dict"],
        strict=True
    )

    model = model.to(device)

    print("\nModel loaded successfully.")
    print(f"Architecture: {architecture}")
    print(f"Classification: {classification}")
    print(f"Hidden layers: {hidden_layers}")
    print(f"Hidden sizes: {hidden_sizes}")
    print(
        f"Accumulated epochs: "
        f"{hyperparameters.get('epochs_trained', 0)}"
    )

    return model, hyperparameters


# ==========================================
# SECTION VERIFICATION
# ==========================================

print("\n" + "=" * 55)
print("TRAINING, EVALUATION, SAVING AND LOADING FUNCTIONS")
print("DEFINED SUCCESSFULLY")
print("=" * 55)



TRAINING, EVALUATION, SAVING AND LOADING FUNCTIONS
DEFINED SUCCESSFULLY


## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
# ==========================================
# INPUT VALIDATION
# ==========================================

def read_integer(prompt, default, minimum=0):
    """
    Reads an integer from the user.
    Returns the default value when input is empty.
    """

    while True:
        value = input(f"{prompt} [{default}]: ").strip()

        if value == "":
            return default

        try:
            number = int(value)

            if number < minimum:
                print(f"Please enter a value greater than or equal to {minimum}.")
                continue

            return number

        except ValueError:
            print("Invalid input. Please enter an integer.")


def read_float(prompt, default, minimum=0.0):
    """
    Reads a floating-point number from the user.
    """

    while True:
        value = input(f"{prompt} [{default}]: ").strip()

        if value == "":
            return default

        try:
            number = float(value)

            if number < minimum:
                print(f"Please enter a value greater than or equal to {minimum}.")
                continue

            return number

        except ValueError:
            print("Invalid input. Please enter a number.")


def read_yes_no(prompt, default=False):
    """
    Reads a yes/no answer from the user.
    """

    default_text = "s" if default else "n"

    while True:
        value = input(
            f"{prompt} (s/n) [{default_text}]: "
        ).strip().lower()

        if value == "":
            return default

        if value in ("s", "si", "sí", "y", "yes"):
            return True

        if value in ("n", "no"):
            return False

        print("Invalid option. Enter s or n.")


# ==========================================
# CREATE A NEW MODEL
# ==========================================

def create_new_network():
    """
    Creates a new neural network according to user configuration.

    Returns:
        model: Created PyTorch model.
        hyperparameters: Model configuration.
    """

    print("\n" + "=" * 55)
    print("CREAR UNA NUEVA RED NEURONAL")
    print("=" * 55)

    # Select classification type
    print("\nSeleccione el tipo de clasificación:")
    print("1. Fashion MNIST original (10 clases)")
    print("2. Clases simplificadas (4 clases)")

    while True:
        classification_option = input("Opción: ").strip()

        if classification_option in ("1", "2"):
            break

        print("Opción inválida.")

    classification = (
        "original"
        if classification_option == "1"
        else "simplified"
    )

    num_classes = (
        ORIGINAL_CLASSES
        if classification == "original"
        else SIMPLIFIED_CLASSES
    )

    # Select architecture
    print("\nSeleccione la arquitectura:")
    print("1. Rectangular")
    print("2. Convolucional (interpolación lineal)")

    while True:
        architecture_option = input("Opción: ").strip()

        if architecture_option in ("1", "2"):
            break

        print("Opción inválida.")

    architecture = (
        "rectangular"
        if architecture_option == "1"
        else "convolutional"
    )

    # Configure hidden layers
    hidden_layers = read_integer(
        "Número de capas ocultas",
        DEFAULT_HIDDEN_LAYERS,
        minimum=0
    )

    if architecture == "rectangular":

        hidden_neurons = read_integer(
            "Número de neuronas por capa",
            DEFAULT_HIDDEN_NEURONS,
            minimum=1
        )

    else:
        hidden_neurons = DEFAULT_HIDDEN_NEURONS

        print(
            "\nLos tamaños de las capas ocultas "
            "se calcularán mediante interpolación lineal."
        )

    # Configure optional optimizations
    print("\nConfiguración de optimizaciones:")

    use_batch_norm = read_yes_no(
        "¿Habilitar Batch Normalization?",
        DEFAULT_USE_BATCH_NORM
    )

    use_dropout = read_yes_no(
        "¿Habilitar Dropout?",
        DEFAULT_USE_DROPOUT
    )

    if use_dropout:

        while True:
            dropout_rate = read_float(
                "Porcentaje de Dropout",
                DEFAULT_DROPOUT_RATE,
                minimum=0.0
            )

            if dropout_rate < 1.0:
                break

            print("Dropout debe ser menor que 1.")

    else:
        dropout_rate = DEFAULT_DROPOUT_RATE

    # Build model
    model, hidden_sizes = build_model(
        architecture=architecture,
        hidden_layers=hidden_layers,
        hidden_neurons=hidden_neurons,
        num_classes=num_classes,
        use_batch_norm=use_batch_norm,
        use_dropout=use_dropout,
        dropout_rate=dropout_rate
    )

    # Store model configuration
    hyperparameters = create_hyperparameters(
        architecture=architecture,
        hidden_layers=hidden_layers,
        hidden_neurons=hidden_neurons,
        hidden_sizes=hidden_sizes,
        classification=classification,
        epochs_trained=0,
        use_batch_norm=use_batch_norm,
        use_dropout=use_dropout,
        dropout_rate=dropout_rate
    )

    hyperparameters["epochs_accumulated"] = 0
    hyperparameters["batch_size"] = DEFAULT_BATCH_SIZE

    print("\nRed neuronal creada correctamente.")

    print(f"Architecture: {architecture}")
    print(f"Classification: {classification}")
    print(f"Hidden sizes: {hidden_sizes}")
    print(f"Output neurons: {num_classes}")

    return model, hyperparameters


# ==========================================
# SELECT DATA LOADERS
# ==========================================

def get_data_loaders(classification):
    """
    Returns the appropriate training and testing loaders.
    """

    if classification == "original":
        return (
            train_original_loader,
            test_original_loader,
            ORIGINAL_CLASS_NAMES
        )

    elif classification == "simplified":
        return (
            train_simplified_loader,
            test_simplified_loader,
            SIMPLIFIED_CLASS_NAMES
        )

    else:
        raise ValueError("Unknown classification type.")


# ==========================================
# MODEL OPERATIONS MENU
# ==========================================

def model_operations_menu(model, hyperparameters):
    """
    Displays the menu for an active neural network.
    Allows repeated training, testing, saving and returning.
    """

    while True:

        print("\n" + "=" * 55)
        print("MENÚ DE OPERACIONES")
        print("=" * 55)

        
        print("\nNETWORK CONFIGURATION")
        print("-" * 45)

        print(f"Architecture: {hyperparameters['architecture']}")
        print(f"Classification: {hyperparameters['classification']}")

        print(f"Hidden layers: {hyperparameters['hidden_layers']}")
        print(f"Hidden sizes: {hyperparameters['hidden_sizes']}")

        print(
            f"Accumulated epochs: "
            f"{hyperparameters.get('epochs_trained', 0)}"
        )

        print(
            f"Batch Normalization: "
            f"{hyperparameters.get('use_batch_norm', False)}"
        )

        print(
            f"Dropout: "
            f"{hyperparameters.get('use_dropout', False)}"
        )

        print(f"Optimizer: {hyperparameters.get('optimizer', 'Adam')}")
        print(f"Learning rate: {hyperparameters.get('learning_rate', 0.001)}")
        print("-" * 45)

        print("\nSeleccione una operación:")
        print("1. Entrenar la red")
        print("2. Probar la red")
        print("3. Guardar la red")
        print("4. Volver al menú principal")

        option = input("\nOpción (1-4): ").strip()

        # --------------------------------------
        # TRAIN MODEL
        # --------------------------------------

        if option == "1":

            epochs = read_integer(
                "\nNúmero de épocas de entrenamiento",
                5,
                minimum=1
            )

            optimizer_name = input(
                f"Optimizador (Adam/SGD) "
                f"[{hyperparameters.get('optimizer', DEFAULT_OPTIMIZER)}]: "
            ).strip()

            if optimizer_name == "":
                optimizer_name = hyperparameters.get(
                    "optimizer",
                    DEFAULT_OPTIMIZER
                )

            if optimizer_name.lower() not in ("adam", "sgd"):
                print("Optimizador inválido. Utilice Adam o SGD.")
                continue

            optimizer_name = optimizer_name.capitalize()

            learning_rate = read_float(
                "Learning rate",
                hyperparameters.get(
                    "learning_rate",
                    DEFAULT_LEARNING_RATE
                ),
                minimum=0.0
            )

            if learning_rate <= 0:
                print("Learning rate must be positive.")
                continue

            weight_decay = read_float(
                "Weight decay",
                hyperparameters.get(
                    "weight_decay",
                    DEFAULT_WEIGHT_DECAY
                ),
                minimum=0.0
            )

            train_loader, _, _ = get_data_loaders(
                hyperparameters["classification"]
            )

            try:

                loss_history = train_model(
                    model=model,
                    train_loader=train_loader,
                    epochs=epochs,
                    learning_rate=learning_rate,
                    optimizer_name=optimizer_name,
                    weight_decay=weight_decay
                )

                # Update accumulated training information
                accumulated_epochs = (
                    hyperparameters.get("epochs_trained", 0) + epochs
                )

                hyperparameters["epochs_trained"] = accumulated_epochs
                hyperparameters["epochs_accumulated"] = accumulated_epochs
                hyperparameters["optimizer"] = optimizer_name
                hyperparameters["learning_rate"] = learning_rate
                hyperparameters["weight_decay"] = weight_decay

                # Display loss history
                plot_loss_history(loss_history)

            except Exception as error:
                print(f"\nTraining error: {error}")

        # --------------------------------------
        # TEST MODEL
        # --------------------------------------

        elif option == "2":

            _, test_loader, class_names = get_data_loaders(
                hyperparameters["classification"]
            )

            try:

                metrics = evaluate_model(
                    model=model,
                    test_loader=test_loader,
                    class_names=class_names
                )

                print("\n" + "=" * 45)
                print("RESUMEN DE EVALUACIÓN")
                print("=" * 45)

                print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
                print(f"Precision: {metrics['precision'] * 100:.2f}%")
                print(f"Recall:    {metrics['recall'] * 100:.2f}%")

                print("=" * 45)


            except Exception as error:
                print(f"\nEvaluation error: {error}")

        # --------------------------------------
        # SAVE MODEL
        # --------------------------------------

        elif option == "3":

            filename_key = (
                f"{hyperparameters['classification']}_"
                f"{hyperparameters['architecture']}"
            )

            default_path = MODEL_FILES[filename_key]

            print("\nNombre del archivo de destino:")
            print(f"Presione Enter para usar: {default_path}")

            filename = input("Archivo: ").strip()

            if filename == "":
                filename = default_path

            try:

                save_model(
                    model=model,
                    hyperparameters=hyperparameters,
                    path=filename
                )

            except Exception as error:
                print(f"\nSaving error: {error}")

        # --------------------------------------
        # RETURN TO MAIN MENU
        # --------------------------------------

        elif option == "4":
            print("\nVolviendo al menú principal...")
            break

        else:
            print("\nOpción inválida. Intente nuevamente.")


# ==========================================
# MAIN MENU
# ==========================================

def main():
    """
    Main interactive menu.
    Allows creating or loading neural networks.
    """

    print("\n" + "=" * 55)
    print("       FASHION MNIST - PYTORCH")
    print("          TAREA 3")
    print("=" * 55)

    print(f"Device: {DEVICE}")

    while True:

        print("\n" + "=" * 55)
        print("MENÚ PRINCIPAL")
        print("=" * 55)

        print("1. Crear una nueva red neuronal")
        print("2. Cargar una red neuronal desde un archivo")
        print("3. Salir")

        option = input("\nSeleccione una opción (1-3): ").strip()

        # --------------------------------------
        # CREATE NEW NETWORK
        # --------------------------------------

        if option == "1":

            try:

                model, hyperparameters = create_new_network()

                model_operations_menu(
                    model,
                    hyperparameters
                )

            except Exception as error:
                print(f"\nError creating the model: {error}")

        # --------------------------------------
        # LOAD EXISTING NETWORK
        # --------------------------------------

        elif option == "2":

            filepath = input(
                "\nRuta del archivo .pth que desea cargar: "
            ).strip()

            if filepath == "":
                print("Debe indicar una ruta válida.")
                continue

            try:

                model, hyperparameters = load_model(filepath)

                model_operations_menu(
                    model,
                    hyperparameters
                )

            except Exception as error:
                print(f"\nError loading the model: {error}")

        # --------------------------------------
        # EXIT PROGRAM
        # --------------------------------------

        elif option == "3":

            print("\nPrograma finalizado. ¡Hasta luego!")
            break

        else:
            print("\nOpción inválida. Seleccione 1, 2 o 3.")



# ==========================================
# FINAL AUDIT OF THE FOUR SAVED MODELS
# ==========================================

EXPECTED_MODELS = {
    "original_rectangular": {
        "classification": "original",
        "architecture": "rectangular",
        "num_classes": 10,
        "hidden_sizes": [512, 512]
    },
    "original_convolutional": {
        "classification": "original",
        "architecture": "convolutional",
        "num_classes": 10,
        "hidden_sizes": [526, 268]
    },
    "simplified_rectangular": {
        "classification": "simplified",
        "architecture": "rectangular",
        "num_classes": 4,
        "hidden_sizes": [512, 512]
    },
    "simplified_convolutional": {
        "classification": "simplified",
        "architecture": "convolutional",
        "num_classes": 4,
        "hidden_sizes": [524, 264]
    }
}

ALLOWED_LAYERS = (
    nn.Flatten,
    nn.Linear,
    nn.ReLU,
    nn.BatchNorm1d,
    nn.Dropout
)

audit_results = []

print("=" * 65)
print("FINAL AUDIT - FASHION MNIST MODELS")
print("=" * 65)

print(f"Working directory: {Path.cwd()}")
print(f"Dataset directory: {DATA_DIR.resolve()}")


for model_key, expected in EXPECTED_MODELS.items():

    print("\n" + "-" * 65)
    print(f"AUDITING: {model_key}")
    print("-" * 65)

    path = MODEL_FILES[model_key]

    if not path.is_file():
        print(f"FAIL: File not found: {path}")
        audit_results.append((model_key, False))
        continue

    try:

        # Check the checkpoint structure
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if not isinstance(checkpoint, dict):
            raise ValueError("Invalid checkpoint format.")

        if "state_dict" not in checkpoint:
            raise ValueError("Missing state_dict.")

        if "hyperparameters" not in checkpoint:
            raise ValueError("Missing hyperparameters.")

        hyperparameters = checkpoint["hyperparameters"]

        # Check required hyperparameters
        required_keys = [
            "architecture",
            "hidden_layers",
            "hidden_neurons",
            "hidden_sizes",
            "epochs_trained",
            "classification",
            "optimizations"
        ]

        missing_keys = [
            key for key in required_keys
            if key not in hyperparameters
        ]

        if missing_keys:
            raise ValueError(
                f"Missing hyperparameters: {missing_keys}"
            )

        # Compare saved configuration
        if hyperparameters["classification"] != expected["classification"]:
            raise ValueError("Incorrect classification type.")

        if hyperparameters["architecture"] != expected["architecture"]:
            raise ValueError("Incorrect architecture.")

        if list(hyperparameters["hidden_sizes"]) != expected["hidden_sizes"]:
            raise ValueError("Incorrect hidden layer sizes.")

        if hyperparameters["hidden_layers"] != 2:
            raise ValueError("Unexpected number of hidden layers.")

        if hyperparameters["epochs_trained"] != 5:
            raise ValueError("Expected five accumulated epochs.")

        # Check that all saved floating-point parameters are finite
        for key, tensor in checkpoint["state_dict"].items():

            if torch.is_floating_point(tensor):
                if not torch.isfinite(tensor).all():
                    raise ValueError(
                        f"Non-finite parameter detected: {key}"
                    )

        # Reconstruct the model using the loading function
        model, loaded_hyperparameters = load_model(
            path,
            device=torch.device("cpu")
        )

        # Check Sequential architecture
        if not isinstance(model, nn.Sequential):
            raise TypeError("The model is not nn.Sequential.")

        modules = list(model.children())

        if not modules or not isinstance(modules[0], nn.Flatten):
            raise ValueError("The first layer must be nn.Flatten.")

        for layer in modules:
            if not isinstance(layer, ALLOWED_LAYERS):
                raise TypeError(
                    f"Unsupported layer: {type(layer).__name__}"
                )

        # Check hidden and output dimensions
        linear_layers = [
            layer for layer in modules
            if isinstance(layer, nn.Linear)
        ]

        actual_hidden_sizes = [
            layer.out_features
            for layer in linear_layers[:-1]
        ]

        output_neurons = linear_layers[-1].out_features

        if actual_hidden_sizes != expected["hidden_sizes"]:
            raise ValueError("Reconstructed hidden sizes do not match.")

        if output_neurons != expected["num_classes"]:
            raise ValueError("Incorrect number of output neurons.")

        # Verify a real forward pass with test images
        _, test_loader, _ = get_data_loaders(
            expected["classification"]
        )

        images, _ = next(iter(test_loader))

        model.eval()

        with torch.no_grad():
            outputs = model(images[:8])

        if outputs.shape != (8, expected["num_classes"]):
            raise ValueError(
                f"Unexpected output shape: {outputs.shape}"
            )

        if not torch.isfinite(outputs).all():
            raise ValueError("The model produced non-finite outputs.")

        print(f"File: {path.name}")
        print(f"File size: {path.stat().st_size:,} bytes")
        print(f"Classification: {hyperparameters['classification']}")
        print(f"Architecture: {hyperparameters['architecture']}")
        print(f"Hidden sizes: {actual_hidden_sizes}")
        print(f"Output neurons: {output_neurons}")
        print(f"Accumulated epochs: {hyperparameters['epochs_trained']}")
        print(f"Output tensor shape: {tuple(outputs.shape)}")

        print("RESULT: PASS")

        audit_results.append((model_key, True))

    except Exception as error:

        print(f"RESULT: FAIL - {error}")
        audit_results.append((model_key, False))



main()
